In [0]:
# READ LANDED CSVs INTO BRONZE DELTA TABLES WITH AUDIT COLUMNS
from pyspark.sql import functions as F
import uuid, datetime

# a unique id for this ingestion run
# ties bronze rows back to a single batch
BATCH_ID = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%S") + "_" +uuid.uuid4().hex[:8]
CATALOG = "airbnb_obs"
LANDING = "/Volumes/airbnb_obs/bronze/landing"

SOURCES = {
    #logical tables name -> landing file
    "hosts":        "hosts.csv",
    "listings":     "listings.csv",
    "bookings":     "bookings.csv"
}
print("Batch:", BATCH_ID)

In [0]:
# INGESTION LOOP

for table_name, fname in SOURCES.items():
    path = f"{LANDING}/{fname}"

    # read everything as string so bronze never rejects a row on type grounds
    df = (spark.read.option("header", True).option("inferSchema", False).csv(path)) #all column land as string
    # audit columns: who/when/where/ every row came from
    df_audited = (
        df.withColumn("_source_file", F.lit(fname)).withColumn("_ingested_at_utc", F.current_timestamp()).withColumn("_batch_id",  F.lit(BATCH_ID))
    )
    target = f"{CATALOG}.bronze.{table_name}" 
    (df_audited.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target))

    print(f" {target} : {df_audited.count():,} rows")


In [0]:
# VERIFY DATA
for t in SOURCES:
    print(f"\n.bronze.{t}")
    display(spark.table(f"{CATALOG}.bronze.{t}").limit(5))